# 🏥 LUMOS: Multimodal Vertebral BMD Estimation & Osteoporosis Screening
### Complete GPU Execution Pipeline with Automatic Resume Support

This notebook runs the complete LUMOS pipeline on **Google Colab GPU (T4 / V100 / A100)**:
- **Phase 2 (Resumed):** Generates remaining SAM segmentation masks using GPU acceleration (~20–30 min vs 20+ hours on CPU). Automatically skips the ~1,100 already-generated masks.
- **Phase 4:** Crops vertebral ROIs (L1–L4) from AP and Lateral views (~5 min).
- **Phase 5–9:** Multi-view Multi-task Deep Learning training for BMD regression and Osteoporosis classification (~1.5–2 hours).
- **Phase 10:** Grad-CAM explainability visualizations (~5 min).

---

## ⚙️ Step 1: Check GPU
Make sure your Colab runtime is set to GPU: **Runtime -> Change runtime type -> T4 GPU**

In [ ]:
!nvidia-smi

## 📁 Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 📦 Step 3: Set Project Directory
Set the path where you placed or extracted the `LUMOS` folder on your Google Drive.

In [ ]:
import os
import sys

# Point this to where your LUMOS folder is stored on Google Drive
PROJECT_DIR = '/content/drive/MyDrive/LUMOS'

# If you uploaded LUMOS.zip to Google Drive, you can uncomment to unzip to local Colab SSD for maximum speed:
# !unzip -q /content/drive/MyDrive/LUMOS.zip -d /content/
# PROJECT_DIR = '/content/LUMOS'

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(f'Could not find project directory: {PROJECT_DIR}. Please check the path!')

os.chdir(PROJECT_DIR)
print(f'Working directory set to: {os.getcwd()}')
!ls -la

## 🔧 Step 4: Install Dependencies

In [ ]:
!pip install -q timm albumentations opencv-python-headless openpyxl grad-cam scikit-learn seaborn
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: CUDA is not available. Please switch runtime to T4 GPU!')

## 📥 Step 5: Verify SAM Checkpoint
If `sam_vit_b_01ec64.pth` is not already in the LUMOS folder, download it directly:

In [ ]:
import os

sam_path = os.path.join(PROJECT_DIR, 'sam_vit_b_01ec64.pth')
if not os.path.exists(sam_path) or os.path.getsize(sam_path) < 300_000_000:
    print('Downloading SAM ViT-B model weights (375MB)...')
    !wget -q --show-progress -O {sam_path} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
    print('Download complete!')
else:
    print(f'SAM checkpoint verified ({os.path.getsize(sam_path)/(1024*1024):.1f} MB)')

## 🎯 Step 6: Phase 2 — Complete Remaining SAM Masks on GPU
Because your `processed/segmentation/pseudo_labels` already has ~1,061 masks, **the script automatically skips them** and only finishes the remaining ~539 images!
On GPU, this will take only **~20 to 30 minutes**.

In [ ]:
!python phase2_sam_pseudolabeling.py --patients all

## ✂️ Step 7: Phase 4 — Generate Vertebral ROI Dataset
Crops L1, L2, L3, and L4 vertebrae from both AP and Lateral views. Takes ~5–8 minutes.

In [ ]:
!python phase4_generate_roi_dataset.py

## 🧠 Step 8: Phase 5–9 — Train Multi-View Multi-Task BMD Model
Trains the full multi-view network on GPU for 100 epochs with early stopping (~1.5–2 hours on T4 GPU).

In [ ]:
!python phase5_9_train_bmd_model.py

## 🔍 Step 9: Phase 10 — Grad-CAM Explainability Heatmaps
Generates clinical visual explanations for model predictions.

In [ ]:
!python phase10_gradcam_explainability.py

## 🎉 All Done!
Your models, metrics, plots, and checkpoints are stored in `checkpoints/` and `analysis_output/` on your Google Drive.